# 08 — Fine-tuned evaluation (Colab GPU)

Same harness + same 100 scenarios as notebook 06, with `adapter_path` set.
Write `reports/metrics_finetuned.json`, then the base-vs-finetuned comparison
table into `reports/evaluation_report.md` and `docs/EVALUATION.md`.
Report ACTUAL numbers only.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q transformers accelerate bitsandbytes peft 2>&1 | tail -1

# Add repo to path
import sys
sys.path.insert(0, '/content/drive/MyDrive/MaintainAI/code')

# Verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')

In [ ]:
# Load fine-tuned model and run evaluation
import json
from src.slm_service import SLMService, HFBackend
from src.slm_eval import run_harness

# ADAPTER PATH - update this to your trained adapter location
ADAPTER = '/content/drive/MyDrive/MaintainAI/slm/checkpoints/exp_001/final'  # or Hub repo id

svc = SLMService(
    config={'slm': {'model_name': 'Qwen/Qwen2.5-3B-Instruct'}},
    backend=HFBackend('Qwen/Qwen2.5-3B-Instruct', quantization='4bit', adapter_path=ADAPTER)
)

# Load test scenarios (same 100 as base evaluation)
test = [json.loads(l) for l in open('data/slm/test.jsonl')]
print(f'Loaded {len(test)} test scenarios')

def analyze_fn(e):
    raw = svc.backend.generate(e['system'] + '\n' + e['user'])
    from src.slm_service import extract_json
    from src.schemas import SLMAnalysis
    obj = extract_json(raw)
    try:
        SLMAnalysis.model_validate(obj or {})
        return {**obj, 'meta': {'backend': svc.version, 'valid': True}}
    except Exception as ex:
        return {'meta': {'backend': svc.version, 'valid': False, 'reason': str(ex)[:200]}}

# Run harness
m = run_harness(test, analyze_fn)
print(json.dumps(m, indent=1))

# Save metrics
import os
os.makedirs('/content/drive/MyDrive/MaintainAI/reports', exist_ok=True)
with open('/content/drive/MyDrive/MaintainAI/reports/metrics_finetuned.json', 'w') as f:
    json.dump({**m, 'backend': svc.version}, f, indent=1)
print('\n✓ Saved: /content/drive/MyDrive/MaintainAI/reports/metrics_finetuned.json')

## After running:
```bash
# Copy to local repo
cp /content/drive/MyDrive/MaintainAI/reports/metrics_finetuned.json reports/
# Generate comparison report
python scripts/compare_slm.py
```

Then update `docs/EVALUATION.md` with both base and fine-tuned numbers.